# 22 — Geometric Selection and Structured Mesh

**Curriculum slot:** Tier 2, slot 22.  
**Prerequisite:** 09 — Mesh Refinement.

## Purpose

Setting up a **transfinite (structured) mesh** requires assigning node counts to
specific edges and faces. The naive approach — inspecting raw tags or calling
`entities_in_bounding_box` with six coordinates — is noisy and hard to read
back. This notebook introduces `g.model.queries.select()`, a geometric
predicate filter that lets you describe *what you want* rather than *where it
lives in tag-space*.

## What you will learn

- `select(entities, on=...)` — keep entities that lie entirely on a plane.
- `select(entities, crossing=...)` — keep entities that straddle a line or plane.
- Primitive formats: axis-aligned `{'z': 0}`, 2-point line, 3-point plane.
- Stacking: chaining `.select()` calls to progressively narrow a set.
- Applying the result to set transfinite node counts on a 2-D plate.
- Extending to 3-D: selecting faces and volumes by plane.

## Problem

A rectangular plate $5 \times 10$ m. We want two meshes:

```
y
│  (0,10)──────────(5,10)
│    │                │
│    │   5 × 10 m     │
│    │    plate       │
│    │                │
│  (0,0)───────────(5,0)
└──────────────────────── x
```

1. **Unstructured** — Delaunay triangles, global size control.
2. **Structured** — transfinite quads, node counts derived from target
   element size, boundaries selected with `select()`.

## 1. Imports and parameters

In [ ]:
from apeGmsh import apeGmsh

# Geometry
Lx, Ly = 5.0, 10.0   # plate dimensions [m]

# Target element sizes
size_unstruct = 0.5   # unstructured mesh
size_x        = 1.0   # structured: spacing along x
size_y        = 0.5   # structured: spacing along y

## 2. The `select()` API

`g.model.queries.select(entities, on=..., crossing=...)` takes a list of
`(dim, tag)` pairs — the output of `boundary()` or another `select()` call —
and returns a **`Selection`**: a list subclass you can iterate, pass to mesh
methods, or filter again with `.select()`.

### Primitive formats — no imports needed

| Argument | Meaning |
|---|---|
| `{'z': 0}` | axis-aligned plane z = 0 |
| `{'x': 5}` | axis-aligned plane x = 5 |
| `[(x1,y1,z1),(x2,y2,z2)]` | infinite line through 2 points |
| `[(x1,y1,z1),(x2,y2,z2),(x3,y3,z3)]` | infinite plane through 3 points |

### Two predicates

| Predicate | Passes when … |
|---|---|
| `on=` | all bounding-box corners lie within `tol` of the primitive |
| `crossing=` | bounding-box corners exist on *both* sides of the primitive |

In [ ]:
m = apeGmsh(model_name='selection_demo', verbose=False)
m.begin()

surf = m.model.geometry.add_rectangle(0, 0, 0, Lx, Ly, label='plate')

# Get all 4 boundary curves at once
curves = m.model.queries.boundary(surf, oriented=False)
print("All curves:", curves)

# ── on= : curves that lie entirely on a plane ──────────────────────────────
bottom = m.model.queries.select(curves, on={'y': 0})
top    = m.model.queries.select(curves, on={'y': Ly})
left   = m.model.queries.select(curves, on={'x': 0})
right  = m.model.queries.select(curves, on={'x': Lx})

print("bottom:", bottom)   # note the repr tells you how to chain
print("top:   ", top)
print("left:  ", left)
print("right: ", right)

# ── crossing= : curves that straddle a line ─────────────────────────────────
# Which curves cross the horizontal mid-line y=5?
# Answer: the two vertical edges (left and right), both span y ∈ [0,10]
mid_crossers = m.model.queries.select(curves, crossing=[(0, Ly/2, 0), (Lx, Ly/2, 0)])
print("crossing y=Ly/2:", mid_crossers)

m.end()

## 3. Stacking selections

Every `Selection` has a `.select()` method — identical arguments to
`queries.select()`. Chain calls to progressively narrow the set.
This is useful in 3-D when you want, say, *vertical faces on the
boundary x = 0*.

In [ ]:
m = apeGmsh(model_name='stacking_demo', verbose=False)
m.begin()

vol = m.model.geometry.add_box(0, 0, 0, Lx, Lx, Ly, label='box')
faces = m.model.queries.boundary(vol, oriented=False)
print(f"All {len(faces)} faces")

# All faces that straddle z = Ly/2  (the 4 vertical faces)
vertical_faces = m.model.queries.select(faces, crossing={'z': Ly/2})
print("Vertical faces:", vertical_faces)

# From those, the one that also lies on x = 0
left_vert = vertical_faces.select(on={'x': 0})
print("Left vertical face:", left_vert)

# Equivalently as a one-liner
left_vert2 = (
    m.model.queries
     .select(faces, crossing={'z': Ly/2})
     .select(on={'x': 0})
)
print("One-liner match:", left_vert == left_vert2)

m.end()

## 4. Baseline — unstructured (triangle) mesh

The unstructured workflow needs no boundary selection: a global size
target drives Delaunay refinement over the whole domain.

In [ ]:
m = apeGmsh(model_name='plate_tri', verbose=False)
m.begin()

m.model.geometry.add_rectangle(0, 0, 0, Lx, Ly, label='plate')

m.mesh.sizing.set_global_size(max_size=size_unstruct)
m.mesh.generation.generate(dim=2)
m.mesh.viewer()

m.end()

## 5. Structured (transfinite quad) mesh

A transfinite mesh requires:

1. A **node count** on every bounding curve — derived from
   `n = round(length / size) + 1`.
2. Every bounding **surface** marked transfinite.
3. Surfaces **recombined** to quads (otherwise the transfinite grid
   produces triangles).

`select()` replaces the six-coordinate bounding-box query with a
readable geometric description. Compare the two approaches:

```python
# Before select() — noisy
tol  = 1e-6
bottom = m.model.queries.entities_in_bounding_box(
    0, -tol, -tol, Lx, tol, tol, dim=1)

# With select() — reads like a description
bottom = m.model.queries.select(curves, on={'y': 0})
```

In [ ]:
m = apeGmsh(model_name='plate_quad', verbose=False)
m.begin()

surf = m.model.geometry.add_rectangle(0, 0, 0, Lx, Ly, label='plate')
curves = m.model.queries.boundary(surf, oriented=False)

# ── Select boundaries by geometric description ──────────────────────────────
x_curves = m.model.queries.select(curves, on={'y': 0}) + \
            m.model.queries.select(curves, on={'y': Ly})   # bottom + top
y_curves = m.model.queries.select(curves, on={'x': 0}) + \
            m.model.queries.select(curves, on={'x': Lx})   # left + right

# ── Node counts from target size ─────────────────────────────────────────────
nx = round(Lx / size_x) + 1   # nodes along x-direction edges
ny = round(Ly / size_y) + 1   # nodes along y-direction edges
print(f"nx = {nx}  ny = {ny}")

# ── Apply transfinite constraints ────────────────────────────────────────────
m.mesh.structured.set_transfinite_curve(x_curves, nx)
m.mesh.structured.set_transfinite_curve(y_curves, ny)
m.mesh.structured.set_transfinite_surface(surf)
m.mesh.structured.set_recombine(surf, dim=2)

m.mesh.generation.generate(dim=2)
m.mesh.viewer()

m.end()

## 6. 3-D extension — structured hex via `set_transfinite_box`

The full transfinite-hex setup collapses into **one call**:

```python
m.mesh.structured.set_transfinite_box('box', size=1.0)
```

`set_transfinite_box` walks the volume's curves, computes node counts from
the target size, marks every face transfinite + recombined, and marks the
volume transfinite. It accepts either `size=` (target element size) or `n=`
(uniform node count per edge).

This cell also demonstrates the rest of the v1.0.7 selection upgrades:

| Feature | What it does |
|---|---|
| `select('box', dim=2, on={...})` | Resolve label → boundary at `dim` |
| `m.model.queries.plane(z=2.5)` | Build a primitive once, reuse many times |
| `not_on=` / `not_crossing=` | Negation predicates |
| `selection \| selection`, `&`, `-` | Set operations |
| `selection.partition_by()` | Group edges/faces by dominant axis |
| `.to_label() / .to_physical()` | Register a selection chainably |

In [ ]:
m = apeGmsh(model_name='box_hex', verbose=False)
m.begin()

vol = m.model.geometry.add_box(0, 0, 0, Lx, Lx, Ly, label='box')

# v1.0.7 — one call replaces the 15-line _apply_hex helper
m.mesh.structured.set_transfinite_box('box', size=1.0)

m.mesh.generation.generate(dim=3)

# ── select() on faces — using the new plane() factory + label resolution ────
xy_mid = m.model.queries.plane(z=Ly/2)              # define once

bottom_face = m.model.queries.select('box', dim=2, on={'z': 0})
top_face    = m.model.queries.select('box', dim=2, on={'z': Ly})
side_faces  = m.model.queries.select('box', dim=2, crossing=xy_mid)
not_top     = m.model.queries.select('box', dim=2, not_on={'z': Ly})  # negation

print("Bottom face:  ", bottom_face)
print("Top face:     ", top_face)
print("Side faces:   ", side_faces)        # 4 vertical faces
print("Not the top:  ", not_top)           # 5 faces

# ── partition_by — group edges by direction (transfinite-ready) ─────────────
edges  = m.model.queries.select('box', dim=1, on={'z': 0})  # bottom 4 edges
groups = edges.partition_by()
print("Bottom edges by axis:")
for ax, sel in groups.items():
    print(f"  {ax}: {sel}")

# ── Set ops — all faces except the bottom, via difference ───────────────────
all_faces = m.model.queries.select('box', dim=2, not_on={'z': -1})  # all
without_bottom = all_faces - bottom_face
print(f"All except bottom: {without_bottom}")

# ── Chained register: filter → label → physical ──────────────────────────────
(m.model.queries
    .select('box', dim=2, on={'z': 0})
    .to_label('base_face')
    .to_physical('Base'))

print("base_face label:", m.labels.entities('base_face', dim=2))

m.mesh.viewer()
m.end()

## What this unlocks

- **Readable structured mesh scripts.** Boundary selection is a geometric
  description (`on={'z': 0}`) rather than a six-coordinate bounding box.
- **Arbitrary planes.** Pass 3 points to `crossing=` to select entities
  cut by any tilted plane — useful for sliced assemblies and oblique
  interfaces.
- **Chainable narrowing.** A `Selection` is just a list — stack `.select()`
  calls to reach any sub-set without nested loops.
- **Consistent across dims.** The same `select()` call works on points,
  curves, surfaces, and volumes; the predicate is always a signed-distance
  test on bounding-box corners.
- **Slot 23 (multi-region structured mesh)** uses `select()` to assign
  different node counts to each sub-domain of a fragmented assembly.